# Decision tree

IRIS 데이터셋을 활용하여, **각 붓꽃에 대한 정보(꽃잎의 너비와 길이, 꽃받침의 너비와 길이 등)를 바탕으로 Decision Tree 모델을 학습시키고, 이를 통해 붓꽃의 종류를 예측**합니다.
이 과정에서 데이터 전처리, 모델 학습, 평가 등을 포함한 다양한 단계를 거치게 될 것입니다.
해당 실습을 통해 여러분이 Decision Tree의 기본 개념과 작동 방식을 이해할 수 있기를 기대합니다.


이번 실습에서는 이러한 것들을 다뤄보려 합니다:
1. IRIS 데이터 살펴보고 전처리
2. Decision Tree 사용
3. Feature 중요도 판별

주로 사용할 라이브러리들의 정보는 다음과 같습니다:
- sklearn: 머신러닝 라이브러리로, Decision Tree 모델 학습 및 평가에 사용됩니다.
- pandas: 데이터프레임과 같은 자료구조를 통해 구조화된 데이터를 효율적으로 처리하고 분석할 수 있습니다.

In [ ]:
# 필요한 라이브러리 install
!pip install graphviz

In [ ]:
# 필요한 라이브러리 import
import pydotplus
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import export_graphviz
from sklearn.model_selection import train_test_split
from IPython.display import Image
import pandas as pd
from sklearn.metrics import accuracy_score

## 1. 데이터 살펴보기

이번 실습에서 가장 먼저 할 일은 IRIS 데이터셋을 불러오고, 이 데이터셋이 어떤 정보들을 포함하고 있는지 자세히 살펴보는 것입니다. IRIS 데이터셋은 머신러닝과 데이터 과학 분야에서 많이 사용되는 클래식한 데이터셋으로, 다양한 붓꽃 종류와 그 특성에 대한 정보를 담고 있습니다.

IRIS 데이터셋은 세 가지 품종의 붓꽃에 대한 데이터를 포함하고 있습니다:
0: setosa, 1: versicolor, 2: virginica.

<img src="https://repository-images.githubusercontent.com/158275423/9a32d741-51c7-4573-9799-8d933ee642c6" height="240px">

In [ ]:
# IRIS 데이터셋 불러오기
iris = load_iris()
data = pd.DataFrame(data=iris.data, columns=iris.feature_names)
data['target'] = iris.target # 0 : setosa, 1 : versicolor, 2 : virginica

# 데이터셋의 첫 10 행을 출력
data.head(10)

데이터셋을 확인한 결과, 각 붓꽃은 네 가지 특성으로 설명되고 있다는 것을 알 수 있습니다:

1. sepal length (꽃받침 길이)
2. sepal width (꽃받침 너비)
3. petal length (꽃잎 길이)
4. petal width (꽃잎 너비)


이제 데이터들의 통계 수치들을 살펴보겠습니다. pandas 라이브러리의 describe() 함수를 사용해 기초적인 통계랑을 확인할 수 있습니다.

In [ ]:
data.describe()

## 2. Decision Tree 사용

모델을 학습시키는 과정에서는 주어진 데이터를 사용하여 모델이 패턴을 학습할 수 있도록 합니다. 그러나, 모델의 성능을 평가할 때 동일한 데이터를 사용하면, 모델이 실제로 새로운 데이터에 대해 얼마나 잘 예측하는지 평가하기 어려울 것입니다. 그렇기에 우린 데이터를 train, test셋으로 나누어서 학습을 진행하려고 합니다.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target, test_size=0.3, random_state=42)

In [ ]:
# 모델을 포함하고있는 객체를 생성합니다.
classifier = DecisionTreeClassifier(random_state=42)

# 모델을 우리의 데이터셋을 사용해 학습시킵니다.
classifier.fit(X_train, y_train)

In [ ]:
# Decision tree 모델 결과물 출력
export_graphviz(
    classifier,
    out_file='iris_tree.dot',
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    rounded=True,
    filled=True
)

graph = pydotplus.graph_from_dot_file('iris_tree.dot')
Image(graph.create_png())

해당 이미지는 Decision Tree를 시각화 한 것입니다. 각 노드는 조건문으로 이루어져 있으며, 데이터는 조건문을 따라 분기됩니다. 각 노드는 여러 가지 정보를 제공합니다:

- 조건: 예를 들어, "petal length (cm) <= 2.45"는 특정 꽃잎 길이가 2.45cm 이하인지 여부를 묻는 조건입니다.
- gini: 지니 계수(Gini index)는 노드의 불순도를 나타내는 지표입니다. 0에 가까울수록 해당 노드의 데이터가 한 클래스에 속해 있음을 의미합니다.(분류가 잘 이루어졌음)
- samples: 해당 노드에 속하는 샘플의 수입니다.
- value: 각 클래스에 속하는 샘플의 수를 나타냅니다.
- class: 예측된 클래스입니다.

이미지를 확인해보면, 모든 결과물에 대한 gini index가 0이 되었습니다! 저희의 주어진 데이터셋에 대해서는 완벽하게 학습했다고 할 수 있습니다.

하지만 이 상황은 "과적합"의 위험이 있습니다.

=> max_depth를 설정해보겠습니다.

In [ ]:
# max_depth를 임의로 설정(3)
classifier = DecisionTreeClassifier(max_depth=3, random_state=42)

In [ ]:
classifier.fit(X_train, y_train)

In [ ]:
# Decision tree 모델 결과물 출력
export_graphviz(
    classifier,
    out_file='iris_tree.dot',
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    rounded=True,
    filled=True
)

graph = pydotplus.graph_from_dot_file('iris_tree.dot')
Image(graph.create_png())

## 3. 모델 평가


In [ ]:
y_pred = classifier.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

### 📝 중간 실습: 우리가 학습시킨 classifier를 사용해서, 주어진 데이터가 어떤 라벨을 가지는지 예측해봅시다.

1. sepal length (꽃받침 길이): 5.8
2. sepal width (꽃받침 너비): 3.1
3. petal length (꽃잎 길이): 4.4
4. petal width (꽃잎 너비): 1.3

HINT: 기본적으로 데이터는 numpy.array 형태이며, classifier는 여러 데이터가 다같이 들어있는 2D(행렬) 형태의 데이터를 input으로 받습니다.

e.g., numpy array 사용 예제:

```
import numpy as np
data = np.array(원하는 데이터 입력)
```


In [ ]:
# 중간 실습: classifier를 사용해서 주어진 데이터 분류하기




## 4. Feature 중요도 판별

Decision Tree 모델의 장점 중 하나는 각 feature의 중요도를 평가할 수 있다는 점입니다. 특성 중요도를 알게 된다면, 우린 다음과 같은 장점들을 가질 수 있습니다.

1. 모델이 예측을 수행하는 데 있어 어떤 특성이 중요한 역할을 하는지 이해할 수 있습니다.
2. 중요도가 낮은 특성을 제거하여 불필요한 메모리 낭비를 줄일 수 있습니다.
3. 데이터셋에서 목표 특성과 다른 특성 사이의 연관성을 파악할 수 있습니다.

In [ ]:
# 피처 중요도 출력
print("Feature importances:", classifier.feature_importances_)

# 📝 실습 과제: Wine dataset


Wine 데이터셋은 세 가지 종류의 와인을 분류하는 데 사용되는 화학적 분석 데이터입니다. 이 데이터셋은 178개의 샘플과 13개의 화학적 특성으로 구성되어 있으며, 각 샘플은 하나의 와인을 나타냅니다.

해당 데이터셋을 활용하여, 와인의 화학적 분석 데이터(알코올 함량, 말산, 애쉬 등)를 바탕으로 Decision Tree 모델을 학습시키고, 이를 통해 와인의 종류를 분류하는 모델을 만들어봅시다!

In [ ]:
# 이 cell의 코드는 수정하지 않으셔도 됩니다!

# 데이터 로드
from sklearn.datasets import load_wine
wine = load_wine()

data = pd.DataFrame(data=wine.data, columns=wine.feature_names)
data['target'] = wine.target

data.head()

In [ ]:
# 데이터 전처리 (trainset, testset 분류)




In [ ]:
# Decision Tree 학습시키기




In [ ]:
# 학습된 Decision Tree 시각화 하기




In [ ]:
# 앞서 정의한 test set을 사용해서 정확도 구하기 (95% 이상의 정확도를 기대합니다!)




정확도가 95% 이상 나온다면 성공적으로 실습을 수행하신 것입니다.

고생하셨습니다!